# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

Note: Giving a single example output is "single shot" prompting.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

# key to improve the results, is via experimentation and iteration.

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [9]:
#  At inference: gpt calculates the probably next token, so we can force it to respond in JSON format
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    # from string to dict
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog index', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/'},
  {'type': 'external site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'professional network',
   'url': 'https://www.linkedin.com/in/e

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'endpoints product page',
   'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-4.7
Updated
6 days ago
•
28.6k
•
1.22k
MiniMaxAI/MiniMax-M2.1
Updated
2 days ago
•
60k
•
542
Qwen/Qwen-Image-Edit-2511
Updated
6 days ago
•
19.7k
•
522
Qwen/Qwen-Image-Layered
Updated
10 days ago
•
15.6k
•
835
google/functiongemma-270m-it
Updated
11 days ago
•
36.6k
•
687
Browse 2M+ models
Spaces
Running
Featured
3.2k
Wan2.2 Animate
👁
3.2k
Wan2.2 Animate
Running
on
Zero
Featured
637
TRELLIS.2
🏢
637
High-fidelity 3D Generation from images
Running
on
Zero
Featured
327
Qwen Image Layered
🚀
327
Decompose an image into layers 

In [31]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, professional, impactful and punchy brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [32]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

ReadTimeout: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=None)

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the leading AI community dedicated to building the future of machine learning. Serving as a central collaboration platform for the global ML community, Hugging Face enables users to create, discover, explore, and experiment with an extensive collection of models, datasets, and machine learning applications. With a vision to promote an open and ethical AI future, Hugging Face empowers the next generation of ML engineers, scientists, and end users.

---

## What We Offer

- **Extensive Model Repository:** Browse and collaborate on over 2 million open-source models covering text, image, video, audio, and even 3D modalities.
- **Datasets:** Access 500,000+ public datasets curated for various machine learning tasks, frequently updated and maintained by contributors worldwide.
- **Spaces:** Host and interact with machine learning applications directly on the platform, with thousands of diverse and cutting-edge AI-powered apps available.
- **Community & Collaboration:** A vibrant, fast-growing community where ML professionals and enthusiasts share ideas, code, and innovations, fostering continuous learning and development.
- **Open Source Tools:** Leverage the Hugging Face open-source stack to accelerate development and deployment of ML projects.
- **Enterprise Solutions:** Tailored Compute and Enterprise plans provide advanced tools with enterprise-grade security, access control, and scalable resources for teams and organizations.

---

## Company Culture

- **Open and Ethical AI:** Hugging Face champions transparency, openness, and ethical considerations in AI development.
- **Collaborative Learning:** Encourages sharing knowledge and building portfolios through open collaboration.
- **Community Focus:** Centralizes around a dynamic, inclusive community of machine learning researchers, developers, and users worldwide.
- **Innovation-Driven:** Constantly pushing the frontier with impactful models, creative applications, and robust tooling.

---

## Our Customers

Hugging Face serves a broad variety of users including:

- Individual machine learning engineers and data scientists building and sharing their work.
- Research scientists accelerating innovation through collaborative platforms.
- Enterprises seeking secure, scalable AI infrastructure and solutions tailored to their team’s needs.
- AI enthusiasts and educators accessing state-of-the-art models and datasets for learning and experimentation.

---

## Careers at Hugging Face

Join a passionate and innovative team committed to shaping the future of AI. Hugging Face offers opportunities for:

- Machine Learning Engineers & Researchers
- Software Developers & DevOps Specialists
- Product Managers & Designers
- Community Engagement and Developer Relations

Work in an environment that values creativity, openness, and impact, contributing to tools that millions worldwide depend on.

---

## Connect & Explore

- Browse over 2 million machine learning models and datasets: [huggingface.co/models](https://huggingface.co/models)
- Discover interactive AI applications on Spaces.
- Access extensive documentation and community forums for support and knowledge sharing.
- Consider Enterprise options for advanced AI team collaborations.

---

**Hugging Face — The AI community building the future.**  
Explore, build, and grow with us.  

[Sign Up Now](https://huggingface.co/join) | [Learn More](https://huggingface.co)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-5.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # delta content instead of result because it's just the incremental update
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Welcome to Hugging Face: Where AI Gets Its Hug On 🤗

---

## Who We Are
Hugging Face is not just a company; it’s a thriving community that’s building the future of artificial intelligence — one model, dataset, and code commit at a time. Consider us the social hub for machine learning engineers, scientists, and curious AI explorers around the globe.

Our mission? To **democratize good machine learning** — making AI accessible, ethical, open, and downright friendly.

---

## What We Offer

### 1. **Models Galore - Over 2 Million and Counting!**
From language transformers like DistilBERT to cutting-edge image editors and 3D generators, our model library is like an AI candy store. Want to build smarter chatbots, generate art, analyze videos, or build apps that blow minds? We got you.

### 2. **Datasets - The Fuel for Your AI Engine**
Browse half a million+ datasets that cover everything from text and image to audio and 3D data. No more endlessly searching, just find, download, and let your ML magic begin.

### 3. **Spaces - Your Playground for AI Apps**
Host, demo, and share your AI applications in our live Spaces. Whether you're animating characters, swapping faces, or generating high-fidelity 3D images — Spaces is the stage for your AI show.

### 4. **Community - Not Just Users, But Co-Creators**
Join a community of 75,000+ AI enthusiasts and professionals who contribute models, datasets, and ideas. Collaborate, learn, and level up — because innovation happens best together!

### 5. **Enterprise & Compute Solutions**
Got serious AI ambitions? We offer enterprise-grade tools with security, access control, and heavyweight compute power to accelerate your team's projects.

---

## Why Hugging Face? Because We’re Not Your Average AI Company.

- **Open Source Power:** We believe in the power of open collaboration. Our hub invites *anyone* to learn from, contribute to, and grow the community.
- **Multi-Modality, Baby:** Text, images, video, audio, 3D — if it’s AI-friendly, we support it.
- **Build Your Portfolio:** Showcase your work, get recognized, and build your very own ML profile.
- **Ethical AI:** Committed to building an AI future that is open, fair, and ethical.
- **Friendly Vibes:** Our little mascot isn't the only thing warm here — our culture nurtures learning, sharing, and a sense of humor.

---

## Culture & Careers

At Hugging Face, you’re not just joining a workplace; you’re joining a family of passionate, curious, and insanely talented AI pioneers. Our team of nearly 200 members is growing rapidly, and we’re always on the lookout for folks who:

- Love *open-source* and *community-driven* innovation.
- Want to push the boundaries of machine learning.
- Believe that AI should be accessible and ethical.
- Can laugh at AI jokes (bonus points).

If you’re looking to make a real impact while building cool tech in a space that values collaboration and creativity, maybe you should… 🤔 *join us*!  

---

## Our Customers

Whether you're a university researcher, a data scientist at a Fortune 500 company, or a solo AI hobbyist tinkering in your garage — Hugging Face powers your AI dreams with tools that fit any scale or ambition. If you want to move faster, build smarter, and embrace the future of ML, Hugging Face is your launchpad.

---

## Get Started Today!

- Explore 2 million+ open-source models
- Browse 500k+ rich datasets
- Host your AI applications on Spaces
- Connect with thousands of AI enthusiasts worldwide

Visit us at [huggingface.co](https://huggingface.co) and start hugging your AI projects into life! 🤗

---

### P.S. Want to join the revolution?  
Become part of the AI community that’s transforming our world. Remember—we're more than code, we're a hug for humanity's AI future.

---

*Hugging Face: Because Machine Learning Should Feel Like a Friendly Chat.*

In [35]:
stream_brochure("Vitol", "https://www.vitol.com/")

Selecting relevant links for https://www.vitol.com/ by calling gpt-5-nano
Found 23 relevant links


# Vitol  
## Global solutions for the world’s energy markets

For over 50 years, Vitol has served global energy markets—bringing together deep market expertise, physical capabilities, and trading strength to help customers navigate complexity, manage risk, and secure reliable supply.

---

## What Vitol does  
Vitol operates across the energy value chain, combining trading with real-world infrastructure and operational capability.

### Core businesses
- **Crude oil & products**: Trading & distribution, shipping, production, downstream, refining, storage & blending, and business solutions.  
- **Gas**: LNG, LPG, pipeline gas, and **biogas**—supporting gas as a displacement fuel and a key part of the energy transition.  
- **Power**: Power trading in major markets, including **renewables** and gas-fired generation, plus business solutions.  
- **Sustainable Energy Solutions**: Renewables, biogas, carbon, hydrogen, carbon capture and storage, EVs, and business solutions.  
- **Metals**: Included as part of Vitol’s broader global solutions offering.

---

## Active in the energy transition  
Vitol states it is **determined to be an active participant in the energy transition**, reflected in its Sustainable Energy Solutions portfolio spanning renewables, biogas, carbon, hydrogen, EVs, and carbon capture and storage.

---

## People & culture  
Vitol’s focus is on building an environment where people can thrive:
- A workplace designed so **individuals from a diverse range of backgrounds can reach their full potential**  
- Emphasis on **our people** as a core part of how the company works and succeeds

---

## Careers at Vitol  
Vitol offers opportunities for experienced professionals and early talent:
- **Open roles**
- **Our culture** and **Our people** insights
- **Early Careers in London** programs and pathways

---

## Governance & responsibility  
Vitol highlights structured commitments and oversight through:
- **Ethics & compliance**
- **Environmental, Social & Governance (ESG)**
- **Vitol Foundation**

---

## Why Vitol  
- **Scale and longevity** in global energy markets (50+ years)  
- **End-to-end capabilities**—from trading and shipping to storage, refining, power, and sustainable solutions  
- A clear intent to help shape the **energy transition** while meeting today’s energy needs

In [34]:
stream_brochure("Vitol", "https://www.vitol.com/")

Selecting relevant links for https://www.vitol.com/ by calling gpt-5-nano
Found 20 relevant links


# Vitol  
## Global energy solutions — trading strength, infrastructure depth, transition focus

Vitol has served the world’s energy markets for over 50 years, delivering reliable, large-scale solutions across crude oil, products, gas, power and evolving sustainable energy markets. With capabilities spanning trading through to logistics and downstream assets, Vitol connects producers, refiners, utilities, industries and end-markets with speed, scale and expertise.

---

## What Vitol does  
### Crude oil & products  
- Trading & distribution  
- Shipping  
- Production  
- Downstream  
- Refining  
- Storage & blending  
- Business solutions  

### Gas  
Positioned around gas as a key displacement fuel in the energy transition, with activity across:  
- LNG  
- Pipeline gas  
- LPG  
- Biogas  
- Business solutions  

### Power  
A strong and experienced power trading presence in the world’s largest markets, including:  
- Renewables  
- Gas-fired generation  
- Business solutions  

### Sustainable Energy Solutions  
Determined to be an active participant in the energy transition, with focus areas including:  
- Renewables  
- Biogas  
- Carbon  
- Hydrogen  
- Carbon capture and storage  
- EVs  

### Metals  
Active in metals as part of Vitol’s broader “global solutions” offering.

---

## How Vitol creates value  
- **Integrated capability:** from market access and trading to shipping, storage, blending and downstream operations.  
- **Risk-aware execution:** built for fast-moving global markets and complex supply chains.  
- **Transition participation:** practical deployment across gas, renewables, carbon and emerging fuels.

---

## People & culture  
Vitol’s careers message is clear: **it strives to create an environment where individuals from a diverse range of backgrounds can reach their full potential.**  
- Focus on **people and culture** as a core part of “who we are.”  
- Pathways include **open roles** and **Early Careers in London**.

---

## Governance & responsibility  
Vitol highlights structured commitments through:  
- **Ethics & compliance**  
- **Environmental, Social & Governance (ESG)**  
- **Vitol Foundation**

---

## Careers at Vitol  
Whether you’re a professional hire or early-career candidate, Vitol offers opportunities across global energy and transition-focused markets. Explore:  
- **Open roles**  
- **Our culture**  
- **Our people**  
- **Early Careers in London**

---

## Who Vitol is for  
- **Customers:** seeking dependable supply, logistics and market solutions across crude, products, gas, power and low-carbon pathways.  
- **Investors/partners:** looking for scale, integrated infrastructure and active participation in the energy transition.  
- **Recruits:** motivated by high-impact global markets, diverse teams, and a culture designed to help individuals thrive.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>